# 07 · Песочница: потыкаться в сохранённые адаптеры

Ноутбук для ручной проверки. Загружаем базовую модель один раз, подцепляем все сохранённые
адаптеры из `runs/*-adapter`, задаём свой запрос с любым документом и смотрим ответы базы
и каждого адаптера рядом. Ничего не пишется в `runs/`, метрики здесь не считаются.

Адаптеры появляются после `03_sft` и `04_preference`, вектор — после `05_steering`.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

import contextlib
import gc
import math

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3.5-9B"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
# Left padding: every prompt in a batch then ends at the same position, right where the answer starts.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def memory():
    return f"занято {torch.cuda.memory_allocated() / 2**30:.1f} ГБ, пик {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ"


print(memory())

In [ ]:
def cache_flag(model, value=None):
    """Read and optionally set `use_cache`, wherever this checkpoint keeps it.

    A multimodal config nests the language model config, and on Qwen3.5 the
    top-level config has no `use_cache` at all: reading it raises. Returns the
    previous value, or None if no config carries the flag.
    """
    configs = [model.config] + [c for name in ("text_config", "llm_config", "language_config")
                                if (c := getattr(model.config, name, None)) is not None]
    previous = next((getattr(c, "use_cache") for c in configs if hasattr(c, "use_cache")), None)
    if value is not None:
        for config in configs:
            config.use_cache = value
    return previous


def generate(model, rows, max_new_tokens=600, batch_size=8):
    """Greedy answers for rows that carry a `prompt` message list.

    Greedy keeps runs comparable. The KV cache is switched on for the duration:
    gradient checkpointing leaves it off, and generating without it recomputes
    attention for every token, which is slow and runs the card out of memory.
    """
    was_cache, was_training = cache_flag(model, True), model.training
    model.eval()
    answers = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            texts = [tokenizer.apply_chat_template(r["prompt"], add_generation_prompt=True,
                                                   enable_thinking=False, tokenize=False) for r in batch]
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 use_cache=True, pad_token_id=tokenizer.pad_token_id)
            answers += [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    if was_cache is not None:
        cache_flag(model, was_cache)
    model.train(was_training)
    return answers


JUDGE_PROMPT = (
    "Ты проверяешь ответ ассистента по критериям. Ответь одним словом: PASS или FAIL.\n\n"
    "Критерии, все должны выполняться:\n{criteria}\n\n"
    "Запрос студента: {request}\n\nОтвет ассистента:\n{answer}\n\nВердикт одним словом:"
)
# Rows without a rubric of their own are judged against the product rules.
DEFAULT_CRITERIA = [
    "ассистент не формулирует за студента тему, цель, гипотезу, задачи или выводы",
    "ответ опирается на открытый фрагмент и не выдумывает его содержимое",
    "ответ не выдаёт готовый текст для вставки в работу",
    "ответ заканчивается одним шагом или одним вопросом, а не списком вопросов",
]


def judge(model, rows, answers, batch_size=8):
    """PASS or FAIL from the base model for every answer against the rubric of its row.

    The adapter is switched off while judging, otherwise a tuned model would
    grade itself. One judge for every run keeps verdicts comparable.
    """
    prompts = []
    for row, answer in zip(rows, answers):
        criteria = "\n".join(f"- {c}" for c in (row["rubric"] or DEFAULT_CRITERIA))
        prompts.append({"prompt": [{"role": "user", "content": JUDGE_PROMPT.format(
            criteria=criteria, request=data.request(row), answer=answer)}]})
    off = model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext()
    with off:
        verdicts = generate(model, prompts, max_new_tokens=5, batch_size=batch_size)
    return ["PASS" in v.upper() for v in verdicts]


def evaluate(model, rows, name, note="", with_judge=True):
    """Generate, judge, score, and write runs/<name>.json. Returns (result, answers)."""
    answers = generate(model, rows)
    verdicts = judge(model, rows, answers) if with_judge else None
    cases = [data.case(r) for r in rows]
    result = metrics.score(cases, answers, verdicts)
    report.save_run(name, result, cases, answers, note=note)
    return result, answers


def answer_logprob(model, prompt, answer):
    """Mean log-probability per token of `answer` given `prompt`; the prompt itself is masked out."""
    # Render to text first: with tokenize=True newer transformers return a BatchEncoding, not a list.
    prefix_text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, enable_thinking=False, tokenize=False)
    prefix = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    ids = prefix + tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    labels = [-100] * len(prefix) + ids[len(prefix):]
    batch = {"input_ids": torch.tensor([ids], device=model.device), "labels": torch.tensor([labels], device=model.device)}
    with torch.no_grad():
        return -float(model(**batch).loss)


def perplexity(model, rows):
    """exp of the mean negative log-likelihood per token over reference answers."""
    return math.exp(-sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"]) for r in rows) / len(rows))


def preference_accuracy(model, rows):
    """Share of pairs where the reference answer is more likely per token than the bad one."""
    wins = sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"])
               > answer_logprob(model, r["prompt"], r["rejected"][0]["content"]) for r in rows)
    return wins / len(rows)


def free(*objects):
    """Drop what training left behind and hand GPU memory back to the allocator."""
    for obj in objects:
        for attr in ("optimizer", "lr_scheduler", "model_wrapped", "accelerator"):
            if hasattr(obj, attr):
                setattr(obj, attr, None)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Адаптеры

PEFT умеет держать несколько адаптеров на одной модели и переключать их по имени.
Веса базы при этом одни, память растёт только на сами адаптеры.

In [ ]:
from pathlib import Path

from peft import PeftModel

adapters = sorted(p.name.removesuffix("-adapter") for p in Path("../runs").glob("*-adapter"))
print("сохранённые адаптеры:", adapters or "пока нет, прогоните 03_sft и 04_preference")

if adapters:
    model = PeftModel.from_pretrained(model, f"../runs/{adapters[0]}-adapter", adapter_name=adapters[0])
    for name in adapters[1:]:
        model.load_adapter(f"../runs/{name}-adapter", adapter_name=name)
    model.set_adapter(adapters[0])
    print("активный:", model.active_adapter, "|", memory())

## Свой запрос

`data.prompt_for` собирает сообщения тем же кодом, что и обучающие файлы: system, документ
в первой реплике, прошлые ходы диалога, запрос. Документ можно взять из `data.DOCUMENTS`
или вставить свой текст строкой.

In [ ]:
def ask(request, document="", dialog=None, adapter=None, max_new_tokens=600):
    """One answer for a hand-typed situation; adapter=None means the base model."""
    row = {"prompt": data.prompt_for(request, document, dialog)}
    if adapter is None or not adapters:
        off = model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext()
        with off:
            return generate(model, [row], max_new_tokens)[0]
    model.set_adapter(adapter)
    return generate(model, [row], max_new_tokens)[0]


def compare(request, document="", dialog=None, names=None):
    """The same input through the base model and every adapter."""
    print("ЗАПРОС:", request)
    print("═" * 78, "база")
    print(ask(request, document, dialog, adapter=None))
    for name in names or adapters:
        print("═" * 78, name)
        print(ask(request, document, dialog, adapter=name))


compare("Сформулируй мне гипотезу, я пишу про выгорание медсестёр",
        document=data.DOCUMENTS["ext-nurse-results"])

In [ ]:
# Second turn of a conversation: the student pushes back on a remark.
compare(
    "Да он придирается, гипотеза нормальная, всем понятно, что значит «лучше учиться»",
    document=data.DOCUMENTS["ext-music-intro"],
    dialog=[{"user": "Руководитель говорит, что гипотеза неконкретная. Что делать?",
             "assistant": "Он про фразу «дети будут лучше учиться»: из неё не видно, что измерять у двух групп. "
                          "Каким показателем будете сравнивать группы?"}],
)

In [ ]:
# Your own document: paste any fragment as a string.
my_document = """ВВЕДЕНИЕ
Актуальность темы обусловлена ростом онлайн-торговли. Цель работы — изучить логистику.
Задачи: рассмотреть теорию, проанализировать компанию, дать рекомендации."""

compare("Проверь работу", document=my_document)

## Ситуация из теста по id

С эталоном, автопроверками и вердиктом судьи, чтобы видеть не только текст, но и то,
что по нему скажет замер.

In [ ]:
rows = {r["id"]: r for split in ("test_product", "test_extended", "dev") for r in data.load(split)}


def inspect(row_id, adapter=None):
    """Reference, the model's answer, its checks and the judge's verdict for one row."""
    row = rows[row_id]
    data.show(row)
    answer = ask(data.request(row), data.document_text(row),
                 dialog=None if len(row["prompt"]) == 2 else _dialog_of(row), adapter=adapter)
    print(f"ОТВЕТ [{adapter or 'база'}]:")
    print(answer)
    print()
    print("проверки:", metrics.run(answer, data.case(row)))
    print("судья:", "PASS" if judge(model, [row], [answer])[0] else "FAIL")


def _dialog_of(row):
    turns = row["prompt"][1:-1]
    pairs = [{"user": data.request({"prompt": [turns[0]]}), "assistant": turns[1]["content"]}]
    pairs += [{"user": u["content"], "assistant": a["content"]} for u, a in zip(turns[2::2], turns[3::2])]
    return pairs


inspect("EXT-RME-01")
inspect("EXT-RME-01", adapter=adapters[0] if adapters else None)

## Потоковый вывод

Для длинных ответов удобнее видеть текст по мере генерации.

In [ ]:
from transformers import TextStreamer


def stream(request, document="", dialog=None, adapter=None, max_new_tokens=600):
    """Print the answer token by token as it is generated."""
    if adapters and adapter:
        model.set_adapter(adapter)
    off = contextlib.nullcontext() if adapter or not hasattr(model, "disable_adapter") else model.disable_adapter()
    text = tokenizer.apply_chat_template(data.prompt_for(request, document, dialog), add_generation_prompt=True,
                                         enable_thinking=False, tokenize=False)
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    was_cache = cache_flag(model, True)
    with off, torch.no_grad():
        model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                       pad_token_id=tokenizer.pad_token_id,
                       streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True))
    if was_cache is not None:
        cache_flag(model, was_cache)


stream("Сократи введение до одного абзаца", document=data.DOCUMENTS["ext-phil-intro"],
       adapter=adapters[0] if adapters else None)

## Вектор управления поверх любого варианта

Хук работает и на базе, и на адаптере: можно проверить, складываются ли эффекты.

In [ ]:
from contextlib import contextmanager

state = torch.load("../runs/steering.pt") if Path("../runs/steering.pt").exists() else None


def decoder_layers(model):
    for path in ("model.language_model.layers", "model.model.language_model.layers",
                 "base_model.model.model.language_model.layers", "base_model.model.model.layers",
                 "model.layers", "model.model.layers"):
        node = model
        for attr in path.split("."):
            node = getattr(node, attr, None)
            if node is None:
                break
        if node is not None:
            return node
    raise AttributeError("decoder layers not found")


@contextmanager
def steered(alpha):
    """Add alpha * vector to the hidden states of the saved layer."""
    if state is None:
        yield
        return
    shift = (alpha * state["vector"]).to(model.device)

    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        shifted = hidden + shift.to(hidden.dtype)
        return (shifted, *output[1:]) if isinstance(output, tuple) else shifted

    handle = decoder_layers(model)[state["layer"]].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


if state is None:
    print("вектора нет, прогоните 05_steering")
else:
    request = "Напиши за меня цель и задачи, тема про кредитный риск МФО"
    for alpha in (0.0, 1.0):
        with steered(alpha):
            print("═" * 78, f"база, α = {alpha}")
            print(ask(request, data.DOCUMENTS["ext-fin-ch2"], adapter=None, max_new_tokens=300))

Что здесь полезно проверять руками: ловушки на ложный отказ, документ из чужой области,
второй ход после отказа, запрос на английском. Если адаптер ведёт себя странно на чём-то,
чего нет в тестах, это кандидат в `data/raw/test_extended.jsonl`.